In [ ]:
import os
import sys
import random
from pathlib import Path

ROOT = "/home/wangxc1117/STDK_ADMM"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
import optuna

torch.set_default_dtype(torch.float32)
optuna.logging.set_verbosity(optuna.logging.WARNING)

from examples.baselines.timesplit import *
from examples.baselines.stdk.st_interp import create_model
from spatial_adapter.models.spatial_adapter import (
    SpatialAdapter,
    SpatialAdapterConfig,
    ADMMConfig,
    TrainingConfig,
    BasisConfig,
)
from spatial_adapter.models.trend_model import TrendModel
from spatial_adapter.models.spatial_basis_learner import SpatialBasisLearner

SEED = 123

WEATHER2K_NPY = Path("/home/wangxc1117/Weather2K/weather2k.npy")
TARGET_VAR_IDX = 4
LAT_IDX = 0
LON_IDX = 1
T_KEEP = 100

EPOCHS = 350
BATCH_SIZE = 512
LR = 1e-3
WEIGHT_DECAY = 1e-5

SPACE_RATIO_KEEP = 0.1

TRAIN_RATIO_TIME = 0.1
VAL_RATIO_TIME = 0.1
TEST_RATIO_TIME = 0.8

GNA_BATCH_SIZE = 64
N_TRIALS = 50
TAU_MIN = 1e-8
TAU_MAX = 1e4

K_FIXED = 40
N_RUNS_FIXED_K = 10

RESULT_DIR = Path("./weather2k")
REPEAT_DIR = RESULT_DIR / (
    f"var{TARGET_VAR_IDX}_tkeep{T_KEEP}_k_{K_FIXED}_fixedspace{SPACE_RATIO_KEEP}_time_train{TRAIN_RATIO_TIME}_val{VAL_RATIO_TIME}_test{TEST_RATIO_TIME}"
)
REPEAT_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV = REPEAT_DIR / f"k_{K_FIXED}_repeat_runs_summary.csv"

PRED_DIR = REPEAT_DIR / "saved_predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

TRIAL_DIR = REPEAT_DIR / "trial_results"
TRIAL_DIR.mkdir(parents=True, exist_ok=True)

PHI_DIR = REPEAT_DIR / "saved_phi"
PHI_DIR.mkdir(parents=True, exist_ok=True)

ALL_TRIAL_CSV = TRIAL_DIR / "all_trial_results.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device, flush=True)


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_weather2k_as_long_df(
    npy_path,
    target_var_idx,
    lat_idx,
    lon_idx,
    t_keep=None,
    normalize_xy=True,
):
    arr = np.load(str(npy_path)).astype(np.float32)

    if arr.ndim != 3:
        raise ValueError(f"Expected arr.ndim == 3 (S, V, T), got shape={arr.shape}")

    S, V, T_full = arr.shape

    if not (0 <= lat_idx < V and 0 <= lon_idx < V and 0 <= target_var_idx < V):
        raise ValueError(
            f"Bad index: lat_idx={lat_idx}, lon_idx={lon_idx}, "
            f"target_var_idx={target_var_idx}, V={V}"
        )

    lat = arr[:, lat_idx, 0].astype(np.float32)
    lon = arr[:, lon_idx, 0].astype(np.float32)
    y_full = arr[:, target_var_idx, :].astype(np.float32)

    if t_keep is not None:
        if t_keep <= 0 or t_keep > T_full:
            raise ValueError(f"t_keep must be in [1, {T_full}], got {t_keep}")
        y_use = y_full[:, -t_keep:]
        T = t_keep
    else:
        y_use = y_full
        T = T_full

    if normalize_xy:
        lon_min, lon_max = float(np.min(lon)), float(np.max(lon))
        lat_min, lat_max = float(np.min(lat)), float(np.max(lat))
        x = ((lon - lon_min) / (lon_max - lon_min + 1e-12)).astype(np.float32)
        y = ((lat - lat_min) / (lat_max - lat_min + 1e-12)).astype(np.float32)
    else:
        x = lon.astype(np.float32)
        y = lat.astype(np.float32)

    t = np.arange(T, dtype=np.int64)

    xx = np.repeat(x, T)
    yy = np.repeat(y, T)
    tt = np.tile(t, S)
    zz = y_use.reshape(-1).astype(np.float32)

    df = pd.DataFrame({
        "x": xx,
        "y": yy,
        "t": tt,
        "z": zz,
    })

    z_np = df["z"].to_numpy(np.float32)
    ok = np.isfinite(z_np)
    df = df.loc[ok].reset_index(drop=True)

    t_np = df["t"].to_numpy(np.float32)
    df["t_norm"] = (t_np - t_np.min()) / (t_np.max() - t_np.min() + 1e-12)

    return df


def new_trend_basis_fixed(n_locations: int):
    trend = TrendModel(
        num_continuous_features=1,
        hidden_layer_sizes=[],
        n_locations=n_locations,
        dropout_rate=0.0,
    )
    basis = SpatialBasisLearner(n_locations, K_FIXED)
    return trend, basis


def adapter_diagnostics(trainer, cont: torch.Tensor, y_true: torch.Tensor):
    with torch.no_grad():
        mu = trainer.trend(cont.to(device))
        residual = y_true.to(device) - mu
        Phi = trainer.basis.basis
        Omega = trainer.omega

        coeff = residual @ Phi
        residual_hat = coeff @ Phi.T

        recon_mse = torch.mean((residual - residual_hat) ** 2).item()
        smooth_penalty = (trainer.tau1 * torch.sum(Phi * (Omega @ Phi))).item()
        l1_penalty = (trainer.tau2 * torch.sum(torch.abs(Phi))).item()
        total_surrogate = recon_mse + smooth_penalty + l1_penalty

        n_locations, k_basis = Phi.shape
        nk = float(n_locations * k_basis)

        smooth_penalty_per_entry = smooth_penalty / nk
        l1_penalty_per_entry = l1_penalty / nk

        eps = 1e-12
        smooth_over_recon = smooth_penalty_per_entry / max(recon_mse, eps)
        l1_over_recon = l1_penalty_per_entry / max(recon_mse, eps)

    return {
        "recon_mse": float(recon_mse),
        "smooth_penalty": float(smooth_penalty),
        "l1_penalty": float(l1_penalty),
        "total_surrogate": float(total_surrogate),
        "n_locations": int(n_locations),
        "k_basis": int(k_basis),
        "smooth_penalty_per_entry": float(smooth_penalty_per_entry),
        "l1_penalty_per_entry": float(l1_penalty_per_entry),
        "smooth_over_recon": float(smooth_over_recon),
        "l1_over_recon": float(l1_over_recon),
    }


seed_everything(SEED)

config = SpatialAdapterConfig(
    admm=ADMMConfig(
        rho=1.0,
        dual_momentum=0.2,
        max_iters=3000,
        min_outer=20,
        tol=1e-4,
    ),
    training=TrainingConfig(
        lr_mu=1e-2,
        batch_size=GNA_BATCH_SIZE,
        pretrain_epochs=5,
    ),
    basis=BasisConfig(
        phi_every=5,
        phi_freeze=200,
    ),
)

In [ ]:
from spatial_adapter.metrics import rmse_pooled, mae_pooled, r2_pooled, empirical_cov, cov_frob_observed


## 10 seed run

In [ ]:
def run_once_fixed_k(run_seed: int):
    seed_everything(run_seed)

    df_full = load_weather2k_as_long_df(
        npy_path=WEATHER2K_NPY,
        target_var_idx=TARGET_VAR_IDX,
        lat_idx=LAT_IDX,
        lon_idx=LON_IDX,
        t_keep=T_KEEP,
        normalize_xy=True,
    )

    df_run, keep_sites_run, n_sites_full_run = build_fixed_location_subset(
        df=df_full,
        keep_ratio=SPACE_RATIO_KEEP,
        seed=run_seed + 11111,
    )

    df_run["t_norm"] = (
        (df_run["t"] - df_run["t"].min()) /
        (df_run["t"].max() - df_run["t"].min() + 1e-12)
    ).astype(np.float32)

    (
        train_mask_flat_run,
        val_mask_flat_run,
        test_mask_flat_run,
        train_time_idx_run,
        val_time_idx_run,
        test_time_idx_run,
        uniq_t_run,
        n_times_run,
    ) = build_contiguous_time_splits(
        df=df_run,
        train_ratio=TRAIN_RATIO_TIME,
        val_ratio=VAL_RATIO_TIME,
        test_ratio=TEST_RATIO_TIME,
    )

    df_run["z_raw"] = df_run["z"].astype(np.float32)

    z_train_raw = df_run.loc[train_mask_flat_run, "z_raw"].to_numpy(np.float32)
    z_mean_run = float(np.mean(z_train_raw))
    z_sd_run = float(np.std(z_train_raw, ddof=0))
    if z_sd_run < 1e-12:
        z_sd_run = 1.0

    df_run["z"] = (
        (df_run["z_raw"].to_numpy(np.float32) - z_mean_run) / (z_sd_run + 1e-12)
    ).astype(np.float32)

    def to_raw(arr_std: np.ndarray) -> np.ndarray:
        return arr_std * z_sd_run + z_mean_run

    coords_all_run = df_run[["x", "y"]].to_numpy(np.float32)
    t_all_run = df_run["t_norm"].to_numpy(np.float32).reshape(-1, 1)
    y_all_run = df_run["z"].to_numpy(np.float32).reshape(-1, 1)
    X_all_run = np.empty((df_run.shape[0], 0), dtype=np.float32)

    X_train_run = X_all_run[train_mask_flat_run]
    coords_train_run = coords_all_run[train_mask_flat_run]
    t_train_run = t_all_run[train_mask_flat_run]
    y_train_run = y_all_run[train_mask_flat_run]

    train_dataset_run = DictDataset(
        torch.from_numpy(X_train_run),
        torch.from_numpy(coords_train_run),
        torch.from_numpy(t_train_run),
        torch.from_numpy(y_train_run),
    )

    g = torch.Generator()
    g.manual_seed(run_seed + 1000)

    train_loader_run = DataLoader(
        train_dataset_run,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=g,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_fn,
    )

    stdk_config = build_stdk_model_config(
        EPOCHS=EPOCHS,
        LR=LR,
        WEIGHT_DECAY=WEIGHT_DECAY,
        BATCH_SIZE=BATCH_SIZE,
    )

    stdk_run = create_model(
        stdk_config,
        train_coords=coords_train_run,
    ).to(device)

    stdk_run = train_simple_loop(
        model=stdk_run,
        train_loader=train_loader_run,
        device=device,
        config=stdk_config,
    )

    y_hat_all_run = predict_all_simple(
        model=stdk_run,
        X=torch.from_numpy(X_all_run),
        coords=torch.from_numpy(coords_all_run),
        t=torch.from_numpy(t_all_run),
        batch_size=BATCH_SIZE,
        device=device,
    )

    coords_run = coords_all_run
    locs_run, inv_loc_run = np.unique(coords_run, axis=0, return_inverse=True)

    t_to_idx_run = {t: i for i, t in enumerate(uniq_t_run)}

    T_run, N_run = len(uniq_t_run), len(locs_run)
    t_idx_run = np.array([t_to_idx_run[t] for t in df_run["t"].to_numpy()])
    s_idx_run = inv_loc_run

    y_stdk_run = np.full((T_run, N_run), np.nan, np.float32)
    y_true_run = np.full((T_run, N_run), np.nan, np.float32)

    y_stdk_run[t_idx_run, s_idx_run] = y_hat_all_run
    y_true_run[t_idx_run, s_idx_run] = df_run["z"].to_numpy(np.float32)

    residual_true_run = y_true_run - y_stdk_run

    time_feat_run = (
        (uniq_t_run - uniq_t_run.min()) /
        (uniq_t_run.max() - uniq_t_run.min() + 1e-12)
    ).astype(np.float32)

    cont_all_run = (
        torch.from_numpy(time_feat_run)
        .float()
        .unsqueeze(1)
        .repeat(1, N_run)
        .unsqueeze(-1)
    )

    train_cont_train_run = cont_all_run[train_time_idx_run]
    train_y_train_run = torch.from_numpy(
        residual_true_run[train_time_idx_run, :]
    ).float()
    train_idx_train_run = torch.arange(len(train_time_idx_run), dtype=torch.long)

    gna_loader_train_run = DataLoader(
        TensorDataset(train_idx_train_run, train_cont_train_run, train_y_train_run),
        batch_size=min(GNA_BATCH_SIZE, len(train_time_idx_run)),
        shuffle=True,
        drop_last=False,
    )

    residual_true_tensor_all_run = torch.from_numpy(residual_true_run).float()

    seed_phi_dir = PHI_DIR / f"seed_{run_seed}"
    seed_phi_dir.mkdir(parents=True, exist_ok=True)

    def fit_adapter_reconstruct_all_times(tag: str, tau1: float, tau2: float, log_dir: str):
        writer = SummaryWriter(str(REPEAT_DIR / log_dir / f"seed_{run_seed}" / tag))
        trend, basis = new_trend_basis_fixed(N_run)
        trainer = SpatialAdapter(
            trend=trend,
            basis=basis,
            train_loader=gna_loader_train_run,
            val_cont=train_cont_train_run,
            val_y=train_y_train_run,
            locs=locs_run.astype(np.float32),
            config=config,
            device=device,
            writer=writer,
            tau1=float(tau1),
            tau2=float(tau2),
        )
        trainer.pretrain_trend()
        trainer.init_basis_dense()
        trainer.run()

        with torch.no_grad():
            pred_all = trainer.reconstruct(
                cont_all_run.to(device),
                residual_true_tensor_all_run.to(device),
            ).cpu().numpy().astype(np.float32)

            phi_np = trainer.basis.basis.detach().cpu().numpy().astype(np.float32)

        diag_train = adapter_diagnostics(
            trainer,
            train_cont_train_run,
            train_y_train_run,
        )

        diag_all = adapter_diagnostics(
            trainer,
            cont_all_run,
            residual_true_tensor_all_run,
        )

        writer.close()
        return pred_all, diag_train, diag_all, phi_np

    def rmse_subset_std(y_pred_std: np.ndarray, subset_idx: np.ndarray) -> float:
        y_pred_sub = y_pred_std[subset_idx, :]
        y_true_sub = y_true_run[subset_idx, :]
        mask = np.isfinite(y_true_sub) & np.isfinite(y_pred_sub)
        return rmse_pooled(y_true_sub, y_pred_sub, mask)

    def rmse_subset_raw(y_pred_std: np.ndarray, subset_idx: np.ndarray) -> float:
        y_pred_sub_raw = to_raw(y_pred_std[subset_idx, :])
        y_true_sub_raw = to_raw(y_true_run[subset_idx, :])
        mask = np.isfinite(y_true_sub_raw) & np.isfinite(y_pred_sub_raw)
        return rmse_pooled(y_true_sub_raw, y_pred_sub_raw, mask)

    def cov_subset_std(y_pred_std: np.ndarray, subset_idx: np.ndarray) -> float:
        return cov_frob_observed(
            y_true_run[subset_idx, :],
            y_pred_std[subset_idx, :],
        )

    def cov_subset_raw(y_pred_std: np.ndarray, subset_idx: np.ndarray) -> float:
        return cov_frob_observed(
            to_raw(y_true_run[subset_idx, :]),
            to_raw(y_pred_std[subset_idx, :]),
        )

    full_time_idx_run = np.arange(T_run, dtype=np.int32)

    residual_full_unreg_run, diag_train_unreg_run, diag_all_unreg_run, phi_unreg_run = fit_adapter_reconstruct_all_times(
        tag="unreg_tau1_0_tau2_0",
        tau1=0.0,
        tau2=0.0,
        log_dir="logs_unreg",
    )

    np.savez_compressed(
        seed_phi_dir / "unreg_phi.npz",
        phi=phi_unreg_run.astype(np.float32),
        tau1=np.array([0.0], dtype=np.float64),
        tau2=np.array([0.0], dtype=np.float64),
        seed=np.array([run_seed], dtype=np.int32),
    )

    y_final_unreg_run = y_stdk_run + residual_full_unreg_run

    val_rmse_unreg_run = rmse_subset_std(y_final_unreg_run, val_time_idx_run)
    val_rmse_unreg_run_raw = rmse_subset_raw(y_final_unreg_run, val_time_idx_run)

    covfrob_stdk_train_base = cov_subset_std(y_stdk_run, train_time_idx_run)
    covfrob_unreg_train_base = cov_subset_std(y_final_unreg_run, train_time_idx_run)

    covfrob_stdk_val_base = cov_subset_std(y_stdk_run, val_time_idx_run)
    covfrob_unreg_val_base = cov_subset_std(y_final_unreg_run, val_time_idx_run)

    covfrob_stdk_test_base = cov_subset_std(y_stdk_run, test_time_idx_run)
    covfrob_unreg_test_base = cov_subset_std(y_final_unreg_run, test_time_idx_run)

    covfrob_stdk_full_base = cov_subset_std(y_stdk_run, full_time_idx_run)
    covfrob_unreg_full_base = cov_subset_std(y_final_unreg_run, full_time_idx_run)

    covfrob_stdk_train_base_raw = cov_subset_raw(y_stdk_run, train_time_idx_run)
    covfrob_unreg_train_base_raw = cov_subset_raw(y_final_unreg_run, train_time_idx_run)

    covfrob_stdk_val_base_raw = cov_subset_raw(y_stdk_run, val_time_idx_run)
    covfrob_unreg_val_base_raw = cov_subset_raw(y_final_unreg_run, val_time_idx_run)

    covfrob_stdk_test_base_raw = cov_subset_raw(y_stdk_run, test_time_idx_run)
    covfrob_unreg_test_base_raw = cov_subset_raw(y_final_unreg_run, test_time_idx_run)

    covfrob_stdk_full_base_raw = cov_subset_raw(y_stdk_run, full_time_idx_run)
    covfrob_unreg_full_base_raw = cov_subset_raw(y_final_unreg_run, full_time_idx_run)

    trial_cache = {}
    trial_rows = []

    def objective_run(trial: optuna.Trial):
        tau1 = trial.suggest_float("tau1", TAU_MIN, TAU_MAX, log=True)
        tau2 = trial.suggest_float("tau2", TAU_MIN, TAU_MAX, log=True)

        pred_all, diag_train, diag_all, phi_trial = fit_adapter_reconstruct_all_times(
            tag=f"reg_trial_{trial.number:03d}_tau1_{tau1:.2e}_tau2_{tau2:.2e}",
            tau1=tau1,
            tau2=tau2,
            log_dir="logs_reg",
        )

        y_final_reg = y_stdk_run + pred_all

        train_rmse = rmse_subset_std(y_final_reg, train_time_idx_run)
        val_rmse = rmse_subset_std(y_final_reg, val_time_idx_run)
        test_rmse = rmse_subset_std(y_final_reg, test_time_idx_run)
        full_rmse = rmse_subset_std(y_final_reg, full_time_idx_run)

        train_rmse_raw = rmse_subset_raw(y_final_reg, train_time_idx_run)
        val_rmse_raw = rmse_subset_raw(y_final_reg, val_time_idx_run)
        test_rmse_raw = rmse_subset_raw(y_final_reg, test_time_idx_run)
        full_rmse_raw = rmse_subset_raw(y_final_reg, full_time_idx_run)

        covfrob_reg_train = cov_subset_std(y_final_reg, train_time_idx_run)
        covfrob_reg_val = cov_subset_std(y_final_reg, val_time_idx_run)
        covfrob_reg_test = cov_subset_std(y_final_reg, test_time_idx_run)
        covfrob_reg_full = cov_subset_std(y_final_reg, full_time_idx_run)

        covfrob_reg_train_raw = cov_subset_raw(y_final_reg, train_time_idx_run)
        covfrob_reg_val_raw = cov_subset_raw(y_final_reg, val_time_idx_run)
        covfrob_reg_test_raw = cov_subset_raw(y_final_reg, test_time_idx_run)
        covfrob_reg_full_raw = cov_subset_raw(y_final_reg, full_time_idx_run)

        phi_path = seed_phi_dir / f"trial_{trial.number:03d}_phi.npz"
        np.savez_compressed(
            phi_path,
            phi=phi_trial.astype(np.float32),
            tau1=np.array([tau1], dtype=np.float64),
            tau2=np.array([tau2], dtype=np.float64),
            seed=np.array([run_seed], dtype=np.int32),
            trial=np.array([trial.number], dtype=np.int32),
        )

        trial_cache[trial.number] = {
            "tau1": float(tau1),
            "tau2": float(tau2),
            "pred_all": pred_all,
            "diag_train": diag_train,
            "diag_all": diag_all,
            "train_rmse": float(train_rmse),
            "val_rmse": float(val_rmse),
            "test_rmse": float(test_rmse),
            "full_rmse": float(full_rmse),
            "train_rmse_raw": float(train_rmse_raw),
            "val_rmse_raw": float(val_rmse_raw),
            "test_rmse_raw": float(test_rmse_raw),
            "full_rmse_raw": float(full_rmse_raw),
            "covfrob_reg_train": float(covfrob_reg_train),
            "covfrob_reg_val": float(covfrob_reg_val),
            "covfrob_reg_test": float(covfrob_reg_test),
            "covfrob_reg_full": float(covfrob_reg_full),
            "covfrob_reg_train_raw": float(covfrob_reg_train_raw),
            "covfrob_reg_val_raw": float(covfrob_reg_val_raw),
            "covfrob_reg_test_raw": float(covfrob_reg_test_raw),
            "covfrob_reg_full_raw": float(covfrob_reg_full_raw),
            "phi_path": str(phi_path),
        }

        trial_rows.append({
            "seed": int(run_seed),
            "trial": int(trial.number),
            "tau1": float(tau1),
            "tau2": float(tau2),
            "log10_tau1": float(np.log10(tau1)),
            "log10_tau2": float(np.log10(tau2)),

            "train_rmse": float(train_rmse),
            "val_rmse": float(val_rmse),
            "test_rmse": float(test_rmse),
            "full_rmse": float(full_rmse),

            "train_rmse_raw": float(train_rmse_raw),
            "val_rmse_raw": float(val_rmse_raw),
            "test_rmse_raw": float(test_rmse_raw),
            "full_rmse_raw": float(full_rmse_raw),

            "covfrob_stdk_train": float(covfrob_stdk_train_base),
            "covfrob_unreg_train": float(covfrob_unreg_train_base),
            "covfrob_reg_train": float(covfrob_reg_train),

            "covfrob_stdk_val": float(covfrob_stdk_val_base),
            "covfrob_unreg_val": float(covfrob_unreg_val_base),
            "covfrob_reg_val": float(covfrob_reg_val),

            "covfrob_stdk_test": float(covfrob_stdk_test_base),
            "covfrob_unreg_test": float(covfrob_unreg_test_base),
            "covfrob_reg_test": float(covfrob_reg_test),

            "covfrob_stdk_full": float(covfrob_stdk_full_base),
            "covfrob_unreg_full": float(covfrob_unreg_full_base),
            "covfrob_reg_full": float(covfrob_reg_full),

            "covfrob_stdk_train_raw": float(covfrob_stdk_train_base_raw),
            "covfrob_unreg_train_raw": float(covfrob_unreg_train_base_raw),
            "covfrob_reg_train_raw": float(covfrob_reg_train_raw),

            "covfrob_stdk_val_raw": float(covfrob_stdk_val_base_raw),
            "covfrob_unreg_val_raw": float(covfrob_unreg_val_base_raw),
            "covfrob_reg_val_raw": float(covfrob_reg_val_raw),

            "covfrob_stdk_test_raw": float(covfrob_stdk_test_base_raw),
            "covfrob_unreg_test_raw": float(covfrob_unreg_test_base_raw),
            "covfrob_reg_test_raw": float(covfrob_reg_test_raw),

            "covfrob_stdk_full_raw": float(covfrob_stdk_full_base_raw),
            "covfrob_unreg_full_raw": float(covfrob_unreg_full_base_raw),
            "covfrob_reg_full_raw": float(covfrob_reg_full_raw),

            "cov_gain_reg_train": float(covfrob_stdk_train_base - covfrob_reg_train),
            "cov_gain_reg_val": float(covfrob_stdk_val_base - covfrob_reg_val),
            "cov_gain_reg_test": float(covfrob_stdk_test_base - covfrob_reg_test),
            "cov_gain_reg_full": float(covfrob_stdk_full_base - covfrob_reg_full),

            "cov_gain_reg_train_raw": float(covfrob_stdk_train_base_raw - covfrob_reg_train_raw),
            "cov_gain_reg_val_raw": float(covfrob_stdk_val_base_raw - covfrob_reg_val_raw),
            "cov_gain_reg_test_raw": float(covfrob_stdk_test_base_raw - covfrob_reg_test_raw),
            "cov_gain_reg_full_raw": float(covfrob_stdk_full_base_raw - covfrob_reg_full_raw),

            "recon_mse_train": float(diag_train["recon_mse"]),
            "smooth_penalty_train": float(diag_train["smooth_penalty"]),
            "l1_penalty_train": float(diag_train["l1_penalty"]),
            "total_surrogate_train": float(diag_train["total_surrogate"]),
            "n_locations": int(diag_train["n_locations"]),
            "k_basis": int(diag_train["k_basis"]),
            "smooth_penalty_per_entry_train": float(diag_train["smooth_penalty_per_entry"]),
            "l1_penalty_per_entry_train": float(diag_train["l1_penalty_per_entry"]),
            "smooth_over_recon_train": float(diag_train["smooth_over_recon"]),
            "l1_over_recon_train": float(diag_train["l1_over_recon"]),

            "recon_mse_all": float(diag_all["recon_mse"]),
            "smooth_penalty_all": float(diag_all["smooth_penalty"]),
            "l1_penalty_all": float(diag_all["l1_penalty"]),
            "total_surrogate_all": float(diag_all["total_surrogate"]),
            "smooth_penalty_per_entry_all": float(diag_all["smooth_penalty_per_entry"]),
            "l1_penalty_per_entry_all": float(diag_all["l1_penalty_per_entry"]),
            "smooth_over_recon_all": float(diag_all["smooth_over_recon"]),
            "l1_over_recon_all": float(diag_all["l1_over_recon"]),

            "z_mean": float(z_mean_run),
            "z_sd": float(z_sd_run),
            "phi_path": str(phi_path),
        })

        print(
            f"[seed {run_seed}] trial {trial.number + 1:03d}/{N_TRIALS:03d} | "
            f"tau1={tau1:.3e} | tau2={tau2:.3e} | "
            f"val_rmse_raw={val_rmse_raw:.6f} | "
            f"val_rmse_std={val_rmse:.6f}",
            flush=True,
        )

        return val_rmse_raw

    study_run = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=run_seed + 2026),
    )
    study_run.optimize(objective_run, n_trials=N_TRIALS, n_jobs=1)

    trial_df = pd.DataFrame(trial_rows).sort_values("trial").reset_index(drop=True)
    trial_csv_path = TRIAL_DIR / f"seed_{run_seed}_trials.csv"
    trial_df.to_csv(trial_csv_path, index=False)

    if "log10_tau1" not in trial_df.columns:
        trial_df["log10_tau1"] = np.log10(trial_df["tau1"])
    if "log10_tau2" not in trial_df.columns:
        trial_df["log10_tau2"] = np.log10(trial_df["tau2"])

    best_run = study_run.best_trial
    tau1_best_run = float(best_run.params["tau1"])
    tau2_best_run = float(best_run.params["tau2"])
    val_rmse_reg_best_run_raw = float(best_run.value)

    residual_full_reg_best_run = trial_cache[best_run.number]["pred_all"]
    diag_train_reg_best_run = trial_cache[best_run.number]["diag_train"]
    diag_all_reg_best_run = trial_cache[best_run.number]["diag_all"]

    phi_reg_best_run = np.load(trial_cache[best_run.number]["phi_path"])["phi"].astype(np.float32)

    np.savez_compressed(
        seed_phi_dir / "reg_best_phi.npz",
        phi=phi_reg_best_run.astype(np.float32),
        tau1=np.array([tau1_best_run], dtype=np.float64),
        tau2=np.array([tau2_best_run], dtype=np.float64),
        seed=np.array([run_seed], dtype=np.int32),
        trial=np.array([best_run.number], dtype=np.int32),
    )

    print(
        f"[seed {run_seed}] best trial = {best_run.number + 1:03d} | "
        f"tau1={tau1_best_run:.3e} | tau2={tau2_best_run:.3e} | "
        f"best_val_rmse_raw={val_rmse_reg_best_run_raw:.6f} | "
        f"best_val_rmse_std={trial_cache[best_run.number]['val_rmse']:.6f}",
        flush=True,
    )

    y_final_reg_best_run = y_stdk_run + residual_full_reg_best_run

    train_mask_run = np.zeros((T_run, N_run), dtype=bool)
    train_mask_run[train_time_idx_run, :] = True

    val_mask_run = np.zeros((T_run, N_run), dtype=bool)
    val_mask_run[val_time_idx_run, :] = True

    test_mask_run = np.zeros((T_run, N_run), dtype=bool)
    test_mask_run[test_time_idx_run, :] = True

    full_mask_run = np.ones((T_run, N_run), dtype=bool)

    def rmse_all_masks_std(y_pred_std: np.ndarray):
        return {
            "full": rmse_pooled(y_true_run, y_pred_std, full_mask_run),
            "train": rmse_pooled(y_true_run, y_pred_std, train_mask_run),
            "val": rmse_pooled(y_true_run, y_pred_std, val_mask_run),
            "test": rmse_pooled(y_true_run, y_pred_std, test_mask_run),
        }

    def rmse_all_masks_raw(y_pred_std: np.ndarray):
        y_true_raw = to_raw(y_true_run)
        y_pred_raw = to_raw(y_pred_std)
        return {
            "full": rmse_pooled(y_true_raw, y_pred_raw, full_mask_run),
            "train": rmse_pooled(y_true_raw, y_pred_raw, train_mask_run),
            "val": rmse_pooled(y_true_raw, y_pred_raw, val_mask_run),
            "test": rmse_pooled(y_true_raw, y_pred_raw, test_mask_run),
        }

    def cov_all_masks_std(y_pred_std: np.ndarray):
        return {
            "full": cov_frob_observed(y_true_run, y_pred_std),
            "train": cov_frob_observed(y_true_run[train_time_idx_run, :], y_pred_std[train_time_idx_run, :]),
            "val": cov_frob_observed(y_true_run[val_time_idx_run, :], y_pred_std[val_time_idx_run, :]),
            "test": cov_frob_observed(y_true_run[test_time_idx_run, :], y_pred_std[test_time_idx_run, :]),
        }

    def cov_all_masks_raw(y_pred_std: np.ndarray):
        y_true_raw = to_raw(y_true_run)
        y_pred_raw = to_raw(y_pred_std)
        return {
            "full": cov_frob_observed(y_true_raw, y_pred_raw),
            "train": cov_frob_observed(y_true_raw[train_time_idx_run, :], y_pred_raw[train_time_idx_run, :]),
            "val": cov_frob_observed(y_true_raw[val_time_idx_run, :], y_pred_raw[val_time_idx_run, :]),
            "test": cov_frob_observed(y_true_raw[test_time_idx_run, :], y_pred_raw[test_time_idx_run, :]),
        }

    rmse_stdk_std = rmse_all_masks_std(y_stdk_run)
    rmse_unreg_std = rmse_all_masks_std(y_final_unreg_run)
    rmse_reg_std = rmse_all_masks_std(y_final_reg_best_run)

    rmse_stdk_raw = rmse_all_masks_raw(y_stdk_run)
    rmse_unreg_raw = rmse_all_masks_raw(y_final_unreg_run)
    rmse_reg_raw = rmse_all_masks_raw(y_final_reg_best_run)

    cov_stdk_std = cov_all_masks_std(y_stdk_run)
    cov_unreg_std = cov_all_masks_std(y_final_unreg_run)
    cov_reg_std = cov_all_masks_std(y_final_reg_best_run)

    cov_stdk_raw = cov_all_masks_raw(y_stdk_run)
    cov_unreg_raw = cov_all_masks_raw(y_final_unreg_run)
    cov_reg_raw = cov_all_masks_raw(y_final_reg_best_run)

    y_true_raw_run = to_raw(y_true_run)
    y_stdk_raw_run = to_raw(y_stdk_run)
    y_unreg_raw_run = to_raw(y_final_unreg_run)
    y_reg_best_raw_run = to_raw(y_final_reg_best_run)

    np.savez_compressed(
        PRED_DIR / f"seed_{run_seed}.npz",
        y_true=y_true_run.astype(np.float32),
        y_stdk=y_stdk_run.astype(np.float32),
        y_unreg=y_final_unreg_run.astype(np.float32),
        y_reg_best=y_final_reg_best_run.astype(np.float32),

        y_true_std=y_true_run.astype(np.float32),
        y_stdk_std=y_stdk_run.astype(np.float32),
        y_unreg_std=y_final_unreg_run.astype(np.float32),
        y_reg_best_std=y_final_reg_best_run.astype(np.float32),

        y_true_raw=y_true_raw_run.astype(np.float32),
        y_stdk_raw=y_stdk_raw_run.astype(np.float32),
        y_unreg_raw=y_unreg_raw_run.astype(np.float32),
        y_reg_best_raw=y_reg_best_raw_run.astype(np.float32),

        phi_unreg=phi_unreg_run.astype(np.float32),
        phi_reg_best=phi_reg_best_run.astype(np.float32),

        train_mask=train_mask_run.astype(bool),
        val_mask=val_mask_run.astype(bool),
        test_mask=test_mask_run.astype(bool),
        full_mask=full_mask_run.astype(bool),

        locs=locs_run.astype(np.float32),
        keep_sites=keep_sites_run.astype(np.int32),
        train_time_idx=train_time_idx_run.astype(np.int32),
        val_time_idx=val_time_idx_run.astype(np.int32),
        test_time_idx=test_time_idx_run.astype(np.int32),
        full_time_idx=full_time_idx_run.astype(np.int32),

        n_locations=np.array([N_run], dtype=np.int32),
        k_basis=np.array([K_FIXED], dtype=np.int32),

        z_mean=np.array([z_mean_run], dtype=np.float64),
        z_sd=np.array([z_sd_run], dtype=np.float64),

        tau1_best=np.array([tau1_best_run], dtype=np.float64),
        tau2_best=np.array([tau2_best_run], dtype=np.float64),

        unreg_recon_mse_train=np.array([diag_train_unreg_run["recon_mse"]], dtype=np.float64),
        unreg_smooth_penalty_train=np.array([diag_train_unreg_run["smooth_penalty"]], dtype=np.float64),
        unreg_l1_penalty_train=np.array([diag_train_unreg_run["l1_penalty"]], dtype=np.float64),
        unreg_total_surrogate_train=np.array([diag_train_unreg_run["total_surrogate"]], dtype=np.float64),
        unreg_smooth_penalty_per_entry_train=np.array([diag_train_unreg_run["smooth_penalty_per_entry"]], dtype=np.float64),
        unreg_l1_penalty_per_entry_train=np.array([diag_train_unreg_run["l1_penalty_per_entry"]], dtype=np.float64),
        unreg_smooth_over_recon_train=np.array([diag_train_unreg_run["smooth_over_recon"]], dtype=np.float64),
        unreg_l1_over_recon_train=np.array([diag_train_unreg_run["l1_over_recon"]], dtype=np.float64),

        reg_best_recon_mse_train=np.array([diag_train_reg_best_run["recon_mse"]], dtype=np.float64),
        reg_best_smooth_penalty_train=np.array([diag_train_reg_best_run["smooth_penalty"]], dtype=np.float64),
        reg_best_l1_penalty_train=np.array([diag_train_reg_best_run["l1_penalty"]], dtype=np.float64),
        reg_best_total_surrogate_train=np.array([diag_train_reg_best_run["total_surrogate"]], dtype=np.float64),
        reg_best_smooth_penalty_per_entry_train=np.array([diag_train_reg_best_run["smooth_penalty_per_entry"]], dtype=np.float64),
        reg_best_l1_penalty_per_entry_train=np.array([diag_train_reg_best_run["l1_penalty_per_entry"]], dtype=np.float64),
        reg_best_smooth_over_recon_train=np.array([diag_train_reg_best_run["smooth_over_recon"]], dtype=np.float64),
        reg_best_l1_over_recon_train=np.array([diag_train_reg_best_run["l1_over_recon"]], dtype=np.float64),

        unreg_recon_mse_all=np.array([diag_all_unreg_run["recon_mse"]], dtype=np.float64),
        unreg_smooth_penalty_all=np.array([diag_all_unreg_run["smooth_penalty"]], dtype=np.float64),
        unreg_l1_penalty_all=np.array([diag_all_unreg_run["l1_penalty"]], dtype=np.float64),
        unreg_total_surrogate_all=np.array([diag_all_unreg_run["total_surrogate"]], dtype=np.float64),
        unreg_smooth_penalty_per_entry_all=np.array([diag_all_unreg_run["smooth_penalty_per_entry"]], dtype=np.float64),
        unreg_l1_penalty_per_entry_all=np.array([diag_all_unreg_run["l1_penalty_per_entry"]], dtype=np.float64),
        unreg_smooth_over_recon_all=np.array([diag_all_unreg_run["smooth_over_recon"]], dtype=np.float64),
        unreg_l1_over_recon_all=np.array([diag_all_unreg_run["l1_over_recon"]], dtype=np.float64),

        reg_best_recon_mse_all=np.array([diag_all_reg_best_run["recon_mse"]], dtype=np.float64),
        reg_best_smooth_penalty_all=np.array([diag_all_reg_best_run["smooth_penalty"]], dtype=np.float64),
        reg_best_l1_penalty_all=np.array([diag_all_reg_best_run["l1_penalty"]], dtype=np.float64),
        reg_best_total_surrogate_all=np.array([diag_all_reg_best_run["total_surrogate"]], dtype=np.float64),
        reg_best_smooth_penalty_per_entry_all=np.array([diag_all_reg_best_run["smooth_penalty_per_entry"]], dtype=np.float64),
        reg_best_l1_penalty_per_entry_all=np.array([diag_all_reg_best_run["l1_penalty_per_entry"]], dtype=np.float64),
        reg_best_smooth_over_recon_all=np.array([diag_all_reg_best_run["smooth_over_recon"]], dtype=np.float64),
        reg_best_l1_over_recon_all=np.array([diag_all_reg_best_run["l1_over_recon"]], dtype=np.float64),
    )

    return {
        "seed": int(run_seed),
        "n_sites_full": int(n_sites_full_run),
        "n_sites_kept": int(len(keep_sites_run)),
        "z_mean": float(z_mean_run),
        "z_sd": float(z_sd_run),

        "tau1_best": float(tau1_best_run),
        "tau2_best": float(tau2_best_run),

        "val_rmse_unreg": float(val_rmse_unreg_run),
        "val_rmse_reg_best": float(trial_cache[best_run.number]["val_rmse"]),
        "val_rmse_unreg_raw": float(val_rmse_unreg_run_raw),
        "val_rmse_reg_best_raw": float(val_rmse_reg_best_run_raw),

        "stdk_full": float(rmse_stdk_std["full"]),
        "unreg_full": float(rmse_unreg_std["full"]),
        "reg_best_full": float(rmse_reg_std["full"]),
        "stdk_train": float(rmse_stdk_std["train"]),
        "unreg_train": float(rmse_unreg_std["train"]),
        "reg_best_train": float(rmse_reg_std["train"]),
        "stdk_val": float(rmse_stdk_std["val"]),
        "unreg_val": float(rmse_unreg_std["val"]),
        "reg_best_val": float(rmse_reg_std["val"]),
        "stdk_test": float(rmse_stdk_std["test"]),
        "unreg_test": float(rmse_unreg_std["test"]),
        "reg_best_test": float(rmse_reg_std["test"]),

        "stdk_full_raw": float(rmse_stdk_raw["full"]),
        "unreg_full_raw": float(rmse_unreg_raw["full"]),
        "reg_best_full_raw": float(rmse_reg_raw["full"]),
        "stdk_train_raw": float(rmse_stdk_raw["train"]),
        "unreg_train_raw": float(rmse_unreg_raw["train"]),
        "reg_best_train_raw": float(rmse_reg_raw["train"]),
        "stdk_val_raw": float(rmse_stdk_raw["val"]),
        "unreg_val_raw": float(rmse_unreg_raw["val"]),
        "reg_best_val_raw": float(rmse_reg_raw["val"]),
        "stdk_test_raw": float(rmse_stdk_raw["test"]),
        "unreg_test_raw": float(rmse_unreg_raw["test"]),
        "reg_best_test_raw": float(rmse_reg_raw["test"]),

        "covfrob_stdk_full": float(cov_stdk_std["full"]),
        "covfrob_unreg_full": float(cov_unreg_std["full"]),
        "covfrob_reg_best_full": float(cov_reg_std["full"]),
        "covfrob_stdk_train": float(cov_stdk_std["train"]),
        "covfrob_unreg_train": float(cov_unreg_std["train"]),
        "covfrob_reg_best_train": float(cov_reg_std["train"]),
        "covfrob_stdk_val": float(cov_stdk_std["val"]),
        "covfrob_unreg_val": float(cov_unreg_std["val"]),
        "covfrob_reg_best_val": float(cov_reg_std["val"]),
        "covfrob_stdk_test": float(cov_stdk_std["test"]),
        "covfrob_unreg_test": float(cov_unreg_std["test"]),
        "covfrob_reg_best_test": float(cov_reg_std["test"]),

        "covfrob_stdk_full_raw": float(cov_stdk_raw["full"]),
        "covfrob_unreg_full_raw": float(cov_unreg_raw["full"]),
        "covfrob_reg_best_full_raw": float(cov_reg_raw["full"]),
        "covfrob_stdk_train_raw": float(cov_stdk_raw["train"]),
        "covfrob_unreg_train_raw": float(cov_unreg_raw["train"]),
        "covfrob_reg_best_train_raw": float(cov_reg_raw["train"]),
        "covfrob_stdk_val_raw": float(cov_stdk_raw["val"]),
        "covfrob_unreg_val_raw": float(cov_unreg_raw["val"]),
        "covfrob_reg_best_val_raw": float(cov_reg_raw["val"]),
        "covfrob_stdk_test_raw": float(cov_stdk_raw["test"]),
        "covfrob_unreg_test_raw": float(cov_unreg_raw["test"]),
        "covfrob_reg_best_test_raw": float(cov_reg_raw["test"]),

        "unreg_recon_mse_train": float(diag_train_unreg_run["recon_mse"]),
        "unreg_smooth_penalty_train": float(diag_train_unreg_run["smooth_penalty"]),
        "unreg_l1_penalty_train": float(diag_train_unreg_run["l1_penalty"]),
        "unreg_total_surrogate_train": float(diag_train_unreg_run["total_surrogate"]),
        "unreg_smooth_penalty_per_entry_train": float(diag_train_unreg_run["smooth_penalty_per_entry"]),
        "unreg_l1_penalty_per_entry_train": float(diag_train_unreg_run["l1_penalty_per_entry"]),
        "unreg_smooth_over_recon_train": float(diag_train_unreg_run["smooth_over_recon"]),
        "unreg_l1_over_recon_train": float(diag_train_unreg_run["l1_over_recon"]),

        "reg_best_recon_mse_train": float(diag_train_reg_best_run["recon_mse"]),
        "reg_best_smooth_penalty_train": float(diag_train_reg_best_run["smooth_penalty"]),
        "reg_best_l1_penalty_train": float(diag_train_reg_best_run["l1_penalty"]),
        "reg_best_total_surrogate_train": float(diag_train_reg_best_run["total_surrogate"]),
        "reg_best_smooth_penalty_per_entry_train": float(diag_train_reg_best_run["smooth_penalty_per_entry"]),
        "reg_best_l1_penalty_per_entry_train": float(diag_train_reg_best_run["l1_penalty_per_entry"]),
        "reg_best_smooth_over_recon_train": float(diag_train_reg_best_run["smooth_over_recon"]),
        "reg_best_l1_over_recon_train": float(diag_train_reg_best_run["l1_over_recon"]),

        "unreg_recon_mse_all": float(diag_all_unreg_run["recon_mse"]),
        "unreg_smooth_penalty_all": float(diag_all_unreg_run["smooth_penalty"]),
        "unreg_l1_penalty_all": float(diag_all_unreg_run["l1_penalty"]),
        "unreg_total_surrogate_all": float(diag_all_unreg_run["total_surrogate"]),
        "unreg_smooth_penalty_per_entry_all": float(diag_all_unreg_run["smooth_penalty_per_entry"]),
        "unreg_l1_penalty_per_entry_all": float(diag_all_unreg_run["l1_penalty_per_entry"]),
        "unreg_smooth_over_recon_all": float(diag_all_unreg_run["smooth_over_recon"]),
        "unreg_l1_over_recon_all": float(diag_all_unreg_run["l1_over_recon"]),

        "reg_best_recon_mse_all": float(diag_all_reg_best_run["recon_mse"]),
        "reg_best_smooth_penalty_all": float(diag_all_reg_best_run["smooth_penalty"]),
        "reg_best_l1_penalty_all": float(diag_all_reg_best_run["l1_penalty"]),
        "reg_best_total_surrogate_all": float(diag_all_reg_best_run["total_surrogate"]),
        "reg_best_smooth_penalty_per_entry_all": float(diag_all_reg_best_run["smooth_penalty_per_entry"]),
        "reg_best_l1_penalty_per_entry_all": float(diag_all_reg_best_run["l1_penalty_per_entry"]),
        "reg_best_smooth_over_recon_all": float(diag_all_reg_best_run["smooth_over_recon"]),
        "reg_best_l1_over_recon_all": float(diag_all_reg_best_run["l1_over_recon"]),
    }


rows_fixed_k = []

for r in range(N_RUNS_FIXED_K):
    run_seed = SEED + r * 1000
    print(
        f"\n===== FIXED K RUN {r+1}/{N_RUNS_FIXED_K} | seed={run_seed} | "
        f"space_keep={SPACE_RATIO_KEEP} | time_train={TRAIN_RATIO_TIME} | "
        f"time_val={VAL_RATIO_TIME} | time_test={TEST_RATIO_TIME} =====",
        flush=True,
    )
    rows_fixed_k.append(run_once_fixed_k(run_seed))

results_fixed_k_df = pd.DataFrame(rows_fixed_k)
results_fixed_k_df.to_csv(SUMMARY_CSV, index=False)

trial_files = sorted(TRIAL_DIR.glob("seed_*_trials.csv"))
if len(trial_files) > 0:
    all_trial_df = pd.concat([pd.read_csv(f) for f in trial_files], ignore_index=True)
    all_trial_df.to_csv(ALL_TRIAL_CSV, index=False)


def mean_sd_repeat(x):
    x = np.asarray(x, dtype=float)
    if len(x) <= 1:
        return np.mean(x), 0.0
    return np.mean(x), np.std(x, ddof=1)


def fmt_pm(m, s):
    return f"{m:.6f} ± {s:.6f}"


print("\n=== FIXED K pooled RMSE summary (across runs; raw main, std secondary) ===", flush=True)
for subset in ["full", "train", "val", "test"]:
    print(f"\n--- {subset} ---", flush=True)
    for name in ["stdk", "unreg", "reg_best"]:
        m_raw, s_raw = mean_sd_repeat(results_fixed_k_df[f"{name}_{subset}_raw"].to_numpy())
        m_std, s_std = mean_sd_repeat(results_fixed_k_df[f"{name}_{subset}"].to_numpy())
        print(
            f"pooled_RMSE({name:8s} | {subset:5s}) | "
            f"raw = {fmt_pm(m_raw, s_raw)} | "
            f"std = {fmt_pm(m_std, s_std)}",
            flush=True,
        )

print("\n=== FIXED K CovFrob summary (across runs; raw main, std secondary) ===", flush=True)
for subset in ["full", "train", "val", "test"]:
    print(f"\n--- {subset} ---", flush=True)
    for name in ["stdk", "unreg", "reg_best"]:
        m_raw, s_raw = mean_sd_repeat(results_fixed_k_df[f"covfrob_{name}_{subset}_raw"].to_numpy())
        m_std, s_std = mean_sd_repeat(results_fixed_k_df[f"covfrob_{name}_{subset}"].to_numpy())
        print(
            f"CovFrob({name:8s} | {subset:5s}) | "
            f"raw = {fmt_pm(m_raw, s_raw)} | "
            f"std = {fmt_pm(m_std, s_std)}",
            flush=True,
        )

tau1_m, tau1_s = mean_sd_repeat(results_fixed_k_df["tau1_best"].to_numpy())
tau2_m, tau2_s = mean_sd_repeat(results_fixed_k_df["tau2_best"].to_numpy())

val_unreg_raw_m, val_unreg_raw_s = mean_sd_repeat(results_fixed_k_df["val_rmse_unreg_raw"].to_numpy())
val_reg_best_raw_m, val_reg_best_raw_s = mean_sd_repeat(results_fixed_k_df["val_rmse_reg_best_raw"].to_numpy())

val_unreg_std_m, val_unreg_std_s = mean_sd_repeat(results_fixed_k_df["val_rmse_unreg"].to_numpy())
val_reg_best_std_m, val_reg_best_std_s = mean_sd_repeat(results_fixed_k_df["val_rmse_reg_best"].to_numpy())

print("\n--- selected tau ---", flush=True)
print(f"tau1_best = {fmt_pm(tau1_m, tau1_s)}", flush=True)
print(f"tau2_best = {fmt_pm(tau2_m, tau2_s)}", flush=True)

print("\n--- validation pooled RMSE ---", flush=True)
print(
    f"val_rmse_unreg    | raw = {fmt_pm(val_unreg_raw_m, val_unreg_raw_s)} | "
    f"std = {fmt_pm(val_unreg_std_m, val_unreg_std_s)}",
    flush=True,
)
print(
    f"val_rmse_reg_best | raw = {fmt_pm(val_reg_best_raw_m, val_reg_best_raw_s)} | "
    f"std = {fmt_pm(val_reg_best_std_m, val_reg_best_std_s)}",
    flush=True,
)

## summary

In [ ]:
def mean_sd_repeat(x):
    x = np.asarray(x, dtype=float)
    if len(x) <= 1:
        return np.mean(x), 0.0
    return np.mean(x), np.std(x, ddof=1)

def fmt_pm(m, s):
    return f"{m:.6f} ± {s:.6f}"

print("\n=== FIXED K pooled RMSE summary (across runs; raw main, std secondary) ===", flush=True)
for subset in ["full", "train", "val", "test"]:
    print(f"\n--- {subset} ---", flush=True)
    for name in ["stdk", "unreg", "reg_best"]:
        m_raw, s_raw = mean_sd_repeat(results_fixed_k_df[f"{name}_{subset}_raw"].to_numpy())
        m_std, s_std = mean_sd_repeat(results_fixed_k_df[f"{name}_{subset}"].to_numpy())
        print(
            f"pooled_RMSE({name:8s} | {subset:5s}) | "
            f"raw = {fmt_pm(m_raw, s_raw)} | "
            f"std = {fmt_pm(m_std, s_std)}",
            flush=True,
        )

print("\n=== FIXED K CovFrob summary (across runs; raw main, std secondary) ===", flush=True)
for subset in ["full", "train", "val", "test"]:
    print(f"\n--- {subset} ---", flush=True)
    for name in ["stdk", "unreg", "reg_best"]:
        m_raw, s_raw = mean_sd_repeat(results_fixed_k_df[f"covfrob_{name}_{subset}_raw"].to_numpy())
        m_std, s_std = mean_sd_repeat(results_fixed_k_df[f"covfrob_{name}_{subset}"].to_numpy())
        print(
            f"CovFrob({name:8s} | {subset:5s}) | "
            f"raw = {fmt_pm(m_raw, s_raw)} | "
            f"std = {fmt_pm(m_std, s_std)}",
            flush=True,
        )

tau1_m, tau1_s = mean_sd_repeat(results_fixed_k_df["tau1_best"].to_numpy())
tau2_m, tau2_s = mean_sd_repeat(results_fixed_k_df["tau2_best"].to_numpy())

val_unreg_raw_m, val_unreg_raw_s = mean_sd_repeat(results_fixed_k_df["val_rmse_unreg_raw"].to_numpy())
val_reg_best_raw_m, val_reg_best_raw_s = mean_sd_repeat(results_fixed_k_df["val_rmse_reg_best_raw"].to_numpy())

val_unreg_std_m, val_unreg_std_s = mean_sd_repeat(results_fixed_k_df["val_rmse_unreg"].to_numpy())
val_reg_best_std_m, val_reg_best_std_s = mean_sd_repeat(results_fixed_k_df["val_rmse_reg_best"].to_numpy())

print("\n--- selected tau ---", flush=True)
print(f"tau1_best = {fmt_pm(tau1_m, tau1_s)}", flush=True)
print(f"tau2_best = {fmt_pm(tau2_m, tau2_s)}", flush=True)

print("\n--- validation pooled RMSE ---", flush=True)
print(
    f"val_rmse_unreg    | raw = {fmt_pm(val_unreg_raw_m, val_unreg_raw_s)} | "
    f"std = {fmt_pm(val_unreg_std_m, val_unreg_std_s)}",
    flush=True,
)
print(
    f"val_rmse_reg_best | raw = {fmt_pm(val_reg_best_raw_m, val_reg_best_raw_s)} | "
    f"std = {fmt_pm(val_reg_best_std_m, val_reg_best_std_s)}",
    flush=True,
)

## diagnostics

In [ ]:
print("\n--- diagnostics: train residual projection (standardized) ---", flush=True)
for name in ["unreg", "reg_best"]:
    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_recon_mse_train"].to_numpy())
    print(f"{name}_recon_mse_train              = {fmt_pm(m, s)}", flush=True)

    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_smooth_penalty_train"].to_numpy())
    print(f"{name}_smooth_penalty_train        = {fmt_pm(m, s)}", flush=True)

    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_l1_penalty_train"].to_numpy())
    print(f"{name}_l1_penalty_train            = {fmt_pm(m, s)}", flush=True)

    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_total_surrogate_train"].to_numpy())
    print(f"{name}_total_surrogate_train       = {fmt_pm(m, s)}", flush=True)
    print("", flush=True)

print("\n--- diagnostics: all-time residual projection (standardized) ---", flush=True)
for name in ["unreg", "reg_best"]:
    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_recon_mse_all"].to_numpy())
    print(f"{name}_recon_mse_all               = {fmt_pm(m, s)}", flush=True)

    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_smooth_penalty_all"].to_numpy())
    print(f"{name}_smooth_penalty_all          = {fmt_pm(m, s)}", flush=True)

    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_l1_penalty_all"].to_numpy())
    print(f"{name}_l1_penalty_all              = {fmt_pm(m, s)}", flush=True)

    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_total_surrogate_all"].to_numpy())
    print(f"{name}_total_surrogate_all         = {fmt_pm(m, s)}", flush=True)
    print("", flush=True)

print("\n--- diagnostics: train residual projection (normalized, standardized base) ---", flush=True)
for name in ["unreg", "reg_best"]:
    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_smooth_penalty_per_entry_train"].to_numpy())
    print(f"{name}_smooth_per_entry_train      = {fmt_pm(m, s)}", flush=True)

    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_l1_penalty_per_entry_train"].to_numpy())
    print(f"{name}_l1_per_entry_train          = {fmt_pm(m, s)}", flush=True)

    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_smooth_over_recon_train"].to_numpy())
    print(f"{name}_smooth_over_recon_train     = {fmt_pm(m, s)}", flush=True)

    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_l1_over_recon_train"].to_numpy())
    print(f"{name}_l1_over_recon_train         = {fmt_pm(m, s)}", flush=True)
    print("", flush=True)

print("\n--- diagnostics: all-time residual projection (normalized, standardized base) ---", flush=True)
for name in ["unreg", "reg_best"]:
    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_smooth_penalty_per_entry_all"].to_numpy())
    print(f"{name}_smooth_per_entry_all        = {fmt_pm(m, s)}", flush=True)

    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_l1_penalty_per_entry_all"].to_numpy())
    print(f"{name}_l1_per_entry_all            = {fmt_pm(m, s)}", flush=True)

    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_smooth_over_recon_all"].to_numpy())
    print(f"{name}_smooth_over_recon_all       = {fmt_pm(m, s)}", flush=True)

    m, s = mean_sd_repeat(results_fixed_k_df[f"{name}_l1_over_recon_all"].to_numpy())
    print(f"{name}_l1_over_recon_all           = {fmt_pm(m, s)}", flush=True)
    print("", flush=True)

print("\n--- fixed space subset info ---", flush=True)
print(f"SPACE_RATIO_KEEP = {SPACE_RATIO_KEEP}", flush=True)
print(f"n_sites_full     = {results_fixed_k_df['n_sites_full'].iloc[0]}", flush=True)
print(f"n_sites_kept     = {results_fixed_k_df['n_sites_kept'].iloc[0]}", flush=True)

print("\n--- time split info ---", flush=True)
print(f"TRAIN_RATIO_TIME = {TRAIN_RATIO_TIME}", flush=True)
print(f"VAL_RATIO_TIME   = {VAL_RATIO_TIME}", flush=True)
print(f"TEST_RATIO_TIME  = {TEST_RATIO_TIME}", flush=True)

print(f"\nSaved summary: {SUMMARY_CSV}", flush=True)
print(f"Saved prediction files dir: {PRED_DIR}", flush=True)
print(f"Saved per-seed trial CSV dir: {TRIAL_DIR}", flush=True)
print(f"Saved phi files dir: {PHI_DIR}", flush=True)
print(f"Saved merged trial CSV: {ALL_TRIAL_CSV}", flush=True)

## heat map

In [ ]:
BEST_BY = "val_rmse_raw"

if ALL_TRIAL_CSV.exists():
    df_all_trials = pd.read_csv(ALL_TRIAL_CSV)

    if df_all_trials.empty:
        print("\nNo trial data found in ALL_TRIAL_CSV.")
    else:
        if "log10_tau1" not in df_all_trials.columns:
            df_all_trials["log10_tau1"] = np.log10(df_all_trials["tau1"])

        if "log10_tau2" not in df_all_trials.columns:
            df_all_trials["log10_tau2"] = np.log10(df_all_trials["tau2"])

        seed_list = sorted(df_all_trials["seed"].dropna().unique().tolist())

        plot_trial_maps(
            df=df_all_trials.copy(),
            output_dir=TRIAL_DIR / "plots_all",
            title_suffix="(All seeds)",
            file_prefix="all_seeds",
            best_by=BEST_BY,
        )

        for seed in seed_list:
            df_seed = df_all_trials[df_all_trials["seed"] == seed].copy()

            plot_trial_maps(
                df=df_seed,
                output_dir=TRIAL_DIR / f"plots_seed_{int(seed)}",
                title_suffix=f"(Seed {int(seed)})",
                file_prefix=f"seed_{int(seed)}",
                best_by=BEST_BY,
            )

        print("\n=== Heat map done ===")
        print(f"Best-point criterion: {BEST_BY}")
        print(f"All-seeds plots saved in: {TRIAL_DIR / 'plots_all'}")
        print("Per-seed plots saved in:")
        for seed in seed_list:
            print(TRIAL_DIR / f"plots_seed_{int(seed)}")
else:
    print("\nSkip heat map: merged trial CSV not found.")

## eigenvector comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUT_DIR = REPEAT_DIR / "eigen_plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def compute_cov(field):
    field = field - np.nanmean(field, axis=0, keepdims=True)
    field = np.nan_to_num(field, nan=0.0, posinf=0.0, neginf=0.0)
    return (field.T @ field) / max(field.shape[0] - 1, 1)


def top_eig(C, k=3):
    vals, vecs = np.linalg.eigh(C)
    idx = np.argsort(vals)[::-1]
    return vals[idx][:k], vecs[:, idx][:, :k]


def plot_eigvec(vec, locs, title, fname, vmin=None, vmax=None):
    plt.figure(figsize=(5, 4))
    sc = plt.scatter(locs[:, 0], locs[:, 1], c=vec, s=8, vmin=vmin, vmax=vmax)
    plt.colorbar(sc)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(fname, dpi=200)
    plt.close()


def eig_alignment(vec_ref, vec_model):
    return np.abs(vec_ref.T @ vec_model)


seed_vis = 123
data = np.load(PRED_DIR / f"seed_{seed_vis}.npz")

y_obs = data["y_true_raw"]
y_stdk = data["y_stdk_raw"]
y_unreg = data["y_unreg_raw"]
y_reg = data["y_reg_best_raw"]
locs = data["locs"]

C_obs = compute_cov(y_obs)
C_stdk = compute_cov(y_stdk)
C_unreg = compute_cov(y_unreg)
C_reg = compute_cov(y_reg)

eig_obs, vec_obs = top_eig(C_obs)
eig_stdk, vec_stdk = top_eig(C_stdk)
eig_unreg, vec_unreg = top_eig(C_unreg)
eig_reg, vec_reg = top_eig(C_reg)

vmin = min(vec_obs.min(), vec_stdk.min(), vec_unreg.min(), vec_reg.min())
vmax = max(vec_obs.max(), vec_stdk.max(), vec_unreg.max(), vec_reg.max())

for i in range(3):
    plot_eigvec(vec_obs[:, i], locs, f"Observed eigvec {i+1}", OUT_DIR / f"obs_eig_{i+1}.png", vmin, vmax)
    plot_eigvec(vec_stdk[:, i], locs, f"STDK eigvec {i+1}", OUT_DIR / f"stdk_eig_{i+1}.png", vmin, vmax)
    plot_eigvec(vec_unreg[:, i], locs, f"Unreg eigvec {i+1}", OUT_DIR / f"unreg_eig_{i+1}.png", vmin, vmax)
    plot_eigvec(vec_reg[:, i], locs, f"Reg eigvec {i+1}", OUT_DIR / f"reg_eig_{i+1}.png", vmin, vmax)

align_stdk = []
align_unreg = []
align_reg = []

for r in range(N_RUNS_FIXED_K):
    seed = SEED + r * 1000
    data = np.load(PRED_DIR / f"seed_{seed}.npz")

    C_obs = compute_cov(data["y_true_raw"])
    C_stdk = compute_cov(data["y_stdk_raw"])
    C_unreg = compute_cov(data["y_unreg_raw"])
    C_reg = compute_cov(data["y_reg_best_raw"])

    _, vec_obs = top_eig(C_obs)
    _, vec_stdk = top_eig(C_stdk)
    _, vec_unreg = top_eig(C_unreg)
    _, vec_reg = top_eig(C_reg)

    A_stdk = eig_alignment(vec_obs, vec_stdk)
    A_unreg = eig_alignment(vec_obs, vec_unreg)
    A_reg = eig_alignment(vec_obs, vec_reg)

    align_stdk.append(np.max(A_stdk, axis=1))
    align_unreg.append(np.max(A_unreg, axis=1))
    align_reg.append(np.max(A_reg, axis=1))

align_stdk = np.array(align_stdk)
align_unreg = np.array(align_unreg)
align_reg = np.array(align_reg)

print("\n=== Eigenvector alignment vs observed covariance (raw; mean ± sd) ===")
for i in range(3):
    m, s = align_stdk[:, i].mean(), align_stdk[:, i].std(ddof=1) if len(align_stdk) > 1 else 0.0
    print(f"eigvec {i+1} | STDK  = {m:.4f} ± {s:.4f}")

    m, s = align_unreg[:, i].mean(), align_unreg[:, i].std(ddof=1) if len(align_unreg) > 1 else 0.0
    print(f"          UNREG = {m:.4f} ± {s:.4f}")

    m, s = align_reg[:, i].mean(), align_reg[:, i].std(ddof=1) if len(align_reg) > 1 else 0.0
    print(f"          REG   = {m:.4f} ± {s:.4f}")

print("\nEigenvector plots saved to:", OUT_DIR)

In [ ]:
import numpy as np

def compute_cov(field):
    field = field - np.nanmean(field, axis=0, keepdims=True)
    field = np.nan_to_num(field, nan=0.0, posinf=0.0, neginf=0.0)
    return (field.T @ field) / max(field.shape[0] - 1, 1)

def eigvals_desc(C):
    vals = np.linalg.eigvalsh(C)
    vals = np.sort(vals)[::-1]
    return vals

seed_vis = 123
data = np.load(PRED_DIR / f"seed_{seed_vis}.npz")

y_obs = data["y_true_raw"]
y_stdk = data["y_stdk_raw"]
y_unreg = data["y_unreg_raw"]
y_reg = data["y_reg_best_raw"]

C_obs = compute_cov(y_obs)
C_stdk = compute_cov(y_stdk)
C_unreg = compute_cov(y_unreg)
C_reg = compute_cov(y_reg)

lam_obs = eigvals_desc(C_obs)
lam_stdk = eigvals_desc(C_stdk)
lam_unreg = eigvals_desc(C_unreg)
lam_reg = eigvals_desc(C_reg)

k = 10
print("\n=== Top eigenvalues (raw; observed covariance reference) ===")
for i in range(k):
    print(
        f"{i+1:02d} | obs={lam_obs[i]:.6f} | "
        f"stdk={lam_stdk[i]:.6f} | "
        f"unreg={lam_unreg[i]:.6f} | "
        f"reg={lam_reg[i]:.6f}"
    )

print("\n=== Explained energy ratio (top 10 / total; raw) ===")
print("obs  :", lam_obs[:10].sum() / lam_obs.sum())
print("stdk :", lam_stdk[:10].sum() / lam_stdk.sum())
print("unreg:", lam_unreg[:10].sum() / lam_unreg.sum())
print("reg  :", lam_reg[:10].sum() / lam_reg.sum())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SEED_VIS = 123

RESULT_DIR = Path("./weather2k")
REPEAT_DIR = RESULT_DIR / (
    f"var{TARGET_VAR_IDX}_tkeep{T_KEEP}_k_{K_FIXED}_fixedspace{SPACE_RATIO_KEEP}_time_train{TRAIN_RATIO_TIME}_val{VAL_RATIO_TIME}_test{TEST_RATIO_TIME}"
)
PRED_DIR = REPEAT_DIR / "saved_predictions"

data = np.load(PRED_DIR / f"seed_{SEED_VIS}.npz")

y_obs = data["y_true_raw"].astype(np.float64)
y_stdk = data["y_stdk_raw"].astype(np.float64)
y_unreg = data["y_unreg_raw"].astype(np.float64)
y_reg = data["y_reg_best_raw"].astype(np.float64)

def compute_cov(field: np.ndarray) -> np.ndarray:
    field = np.asarray(field, dtype=np.float64)
    field = field - np.nanmean(field, axis=0, keepdims=True)
    field = np.nan_to_num(field, nan=0.0, posinf=0.0, neginf=0.0)
    n = field.shape[0]
    if n <= 1:
        raise ValueError("field must have at least 2 time points")
    C = (field.T @ field) / (n - 1)
    C = 0.5 * (C + C.T)
    return C

def eigvals_desc(C: np.ndarray) -> np.ndarray:
    lam = np.linalg.eigvalsh(C)
    lam = np.sort(lam)[::-1]
    return lam

def safe_nonneg_eigs(lam: np.ndarray, tol: float = 1e-10) -> np.ndarray:
    lam = np.asarray(lam, dtype=np.float64).copy()
    lam[np.abs(lam) < tol] = 0.0
    lam = np.maximum(lam, 0.0)
    return lam

def explained_energy_ratio(lam: np.ndarray, k: int = 10) -> float:
    lam = safe_nonneg_eigs(lam)
    s = lam.sum()
    if s <= 0:
        return np.nan
    k = min(k, len(lam))
    return float(lam[:k].sum() / s)

def effective_rank(lam: np.ndarray) -> float:
    lam = safe_nonneg_eigs(lam)
    s = lam.sum()
    if s <= 0:
        return np.nan
    p = lam / s
    p = p[p > 0]
    if len(p) == 0:
        return np.nan
    return float(np.exp(-np.sum(p * np.log(p))))

def normalize_eigs(lam: np.ndarray) -> np.ndarray:
    lam = safe_nonneg_eigs(lam)
    s = lam.sum()
    if s <= 0:
        return lam
    return lam / s

def cumulative_energy(lam: np.ndarray) -> np.ndarray:
    lam = safe_nonneg_eigs(lam)
    s = lam.sum()
    if s <= 0:
        return np.full_like(lam, np.nan, dtype=np.float64)
    return np.cumsum(lam) / s

C_obs = compute_cov(y_obs)
C_stdk = compute_cov(y_stdk)
C_unreg = compute_cov(y_unreg)
C_reg = compute_cov(y_reg)

lam_obs = eigvals_desc(C_obs)
lam_stdk = eigvals_desc(C_stdk)
lam_unreg = eigvals_desc(C_unreg)
lam_reg = eigvals_desc(C_reg)

print("\n=== Eigenvalue diagnostics (raw; observed covariance reference) ===")
for name, lam in {
    "obs": lam_obs,
    "stdk": lam_stdk,
    "unreg": lam_unreg,
    "reg": lam_reg,
}.items():
    lam_clip = safe_nonneg_eigs(lam)
    print(
        f"{name:5s} | "
        f"min_raw={lam.min():.6e} | "
        f"max_raw={lam.max():.6e} | "
        f"sum_clip={lam_clip.sum():.6f} | "
        f"rank_eff={effective_rank(lam):.4f}"
    )

k_show = min(10, len(lam_obs))

print("\n=== Top eigenvalues ===")
for i in range(k_show):
    print(
        f"{i+1:02d} | "
        f"obs={lam_obs[i]:.6f} | "
        f"stdk={lam_stdk[i]:.6f} | "
        f"unreg={lam_unreg[i]:.6f} | "
        f"reg={lam_reg[i]:.6f}"
    )

print("\n=== Tail eigenvalues ===")
for i in range(1, k_show + 1):
    print(
        f"{i:02d} from end | "
        f"obs={lam_obs[-i]:.6f} | "
        f"stdk={lam_stdk[-i]:.6f} | "
        f"unreg={lam_unreg[-i]:.6f} | "
        f"reg={lam_reg[-i]:.6f}"
    )

print("\n=== Explained energy ratio (top 10 / total) ===")
print("obs  :", explained_energy_ratio(lam_obs, 10))
print("stdk :", explained_energy_ratio(lam_stdk, 10))
print("unreg:", explained_energy_ratio(lam_unreg, 10))
print("reg  :", explained_energy_ratio(lam_reg, 10))

print("\n=== Effective rank ===")
print("obs  :", effective_rank(lam_obs))
print("stdk :", effective_rank(lam_stdk))
print("unreg:", effective_rank(lam_unreg))
print("reg  :", effective_rank(lam_reg))

lam_obs_n = normalize_eigs(lam_obs)
lam_stdk_n = normalize_eigs(lam_stdk)
lam_unreg_n = normalize_eigs(lam_unreg)
lam_reg_n = normalize_eigs(lam_reg)

cum_obs = cumulative_energy(lam_obs)
cum_stdk = cumulative_energy(lam_stdk)
cum_unreg = cumulative_energy(lam_unreg)
cum_reg = cumulative_energy(lam_reg)

x = np.arange(1, len(lam_obs) + 1)

plt.figure(figsize=(7, 5))
plt.plot(x, safe_nonneg_eigs(lam_obs), label="Observed")
plt.plot(x, safe_nonneg_eigs(lam_stdk), label="STDK")
plt.plot(x, safe_nonneg_eigs(lam_unreg), label="UNREG")
plt.plot(x, safe_nonneg_eigs(lam_reg), label="REG")
plt.yscale("log")
plt.xlabel("Eigenvalue index")
plt.ylabel("Eigenvalue (log scale)")
plt.title(f"Scree plot (raw, seed={SEED_VIS})")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(x, lam_obs_n, label="Observed")
plt.plot(x, lam_stdk_n, label="STDK")
plt.plot(x, lam_unreg_n, label="UNREG")
plt.plot(x, lam_reg_n, label="REG")
plt.yscale("log")
plt.xlabel("Eigenvalue index")
plt.ylabel("Normalized eigenvalue (log scale)")
plt.title(f"Normalized scree plot (raw, seed={SEED_VIS})")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(x, cum_obs, label="Observed")
plt.plot(x, cum_stdk, label="STDK")
plt.plot(x, cum_unreg, label="UNREG")
plt.plot(x, cum_reg, label="REG")
plt.xlabel("Eigenvalue index")
plt.ylabel("Cumulative explained energy")
plt.title(f"Cumulative energy (raw, seed={SEED_VIS})")
plt.legend()
plt.tight_layout()
plt.show()

## Basis diagnostics (phi analysis)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PHI_DIR = REPEAT_DIR / "saved_phi"
OUT_DIR = REPEAT_DIR / "phi_spectrum_plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)

seed_dirs = sorted(PHI_DIR.glob("seed_*"))

mean_abs_unreg = []
mean_abs_reg = []

singvals_unreg = []
singvals_reg = []

used_seeds = []

for seed_dir in seed_dirs:
    unreg_path = seed_dir / "unreg_phi.npz"
    reg_path = seed_dir / "reg_best_phi.npz"

    if (not unreg_path.exists()) or (not reg_path.exists()):
        continue

    phi_unreg = np.load(unreg_path)["phi"].astype(np.float64)
    phi_reg = np.load(reg_path)["phi"].astype(np.float64)

    mean_abs_unreg.append(np.mean(np.abs(phi_unreg)))
    mean_abs_reg.append(np.mean(np.abs(phi_reg)))

    _, s_unreg, _ = np.linalg.svd(phi_unreg, full_matrices=False)
    _, s_reg, _ = np.linalg.svd(phi_reg, full_matrices=False)

    singvals_unreg.append(s_unreg)
    singvals_reg.append(s_reg)

    used_seeds.append(seed_dir.name)

if len(singvals_unreg) == 0 or len(singvals_reg) == 0:
    raise ValueError(f"No valid phi files found under {PHI_DIR}")

singvals_unreg = np.array(singvals_unreg, dtype=np.float64)
singvals_reg = np.array(singvals_reg, dtype=np.float64)

mean_abs_unreg = np.array(mean_abs_unreg, dtype=np.float64)
mean_abs_reg = np.array(mean_abs_reg, dtype=np.float64)

mean_s_unreg = singvals_unreg.mean(axis=0)
std_s_unreg = singvals_unreg.std(axis=0, ddof=1) if singvals_unreg.shape[0] > 1 else np.zeros_like(mean_s_unreg)

mean_s_reg = singvals_reg.mean(axis=0)
std_s_reg = singvals_reg.std(axis=0, ddof=1) if singvals_reg.shape[0] > 1 else np.zeros_like(mean_s_reg)

x = np.arange(1, len(mean_s_unreg) + 1)

plt.figure(figsize=(6, 4))
plt.plot(x, mean_s_unreg, label="Unreg")
plt.fill_between(
    x,
    mean_s_unreg - std_s_unreg,
    mean_s_unreg + std_s_unreg,
    alpha=0.2,
)
plt.plot(x, mean_s_reg, label="Reg")
plt.fill_between(
    x,
    mean_s_reg - std_s_reg,
    mean_s_reg + std_s_reg,
    alpha=0.2,
)
plt.title("Singular values of phi")
plt.xlabel("Index")
plt.ylabel("Singular value")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "phi_singular_values_mean_sd.png", dpi=200)
plt.show()
plt.close()

plt.figure(figsize=(5, 4))
plt.boxplot([mean_abs_unreg, mean_abs_reg], labels=["Unreg", "Reg"])
plt.title("Mean |phi| comparison")
plt.ylabel("Mean absolute value")
plt.tight_layout()
plt.savefig(OUT_DIR / "phi_mean_abs_boxplot.png", dpi=200)
plt.show()
plt.close()

print("\n=== Phi magnitude ===")
if len(mean_abs_unreg) > 1:
    print(f"Unreg mean ± sd: {np.mean(mean_abs_unreg):.4f} ± {np.std(mean_abs_unreg, ddof=1):.4f}")
else:
    print(f"Unreg mean ± sd: {np.mean(mean_abs_unreg):.4f} ± 0.0000")

if len(mean_abs_reg) > 1:
    print(f"Reg   mean ± sd: {np.mean(mean_abs_reg):.4f} ± {np.std(mean_abs_reg, ddof=1):.4f}")
else:
    print(f"Reg   mean ± sd: {np.mean(mean_abs_reg):.4f} ± 0.0000")

print("\n=== Phi singular values (first 10; mean ± sd) ===")
k_show = min(10, len(mean_s_unreg))
for i in range(k_show):
    su = std_s_unreg[i] if len(mean_abs_unreg) > 1 else 0.0
    sr = std_s_reg[i] if len(mean_abs_reg) > 1 else 0.0
    print(
        f"{i+1:02d} | "
        f"Unreg = {mean_s_unreg[i]:.4f} ± {su:.4f} | "
        f"Reg = {mean_s_reg[i]:.4f} ± {sr:.4f}"
    )

print("\nUsed seeds:")
for s in used_seeds:
    print(s)

print(f"\nSaved phi plots to: {OUT_DIR}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PHI_DIR = REPEAT_DIR / "saved_phi"
OUT_DIR = REPEAT_DIR / "basis_plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_BASIS_TO_PLOT = 3
TARGET_SEED = 123
RANDOM_RECON_SEED = 0

def plot_basis_map(phi_col, locs, title, save_path, vmin=None, vmax=None):
    plt.figure(figsize=(5, 4))
    sc = plt.scatter(
        locs[:, 0],
        locs[:, 1],
        c=phi_col,
        s=12,
        vmin=vmin,
        vmax=vmax,
    )
    plt.colorbar(sc)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()

seed_dirs = sorted(PHI_DIR.glob("seed_*"))

if len(seed_dirs) == 0:
    raise FileNotFoundError(f"No seed folders found in {PHI_DIR}")

pred_file = REPEAT_DIR / "saved_predictions" / f"seed_{TARGET_SEED}.npz"
if not pred_file.exists():
    raise FileNotFoundError(f"Prediction file not found: {pred_file}")

pred_data = np.load(pred_file)
locs = pred_data["locs"]

seed_phi_dir = PHI_DIR / f"seed_{TARGET_SEED}"
if not seed_phi_dir.exists():
    raise FileNotFoundError(f"Seed phi folder not found: {seed_phi_dir}")

unreg_phi_path = seed_phi_dir / "unreg_phi.npz"
reg_phi_path = seed_phi_dir / "reg_best_phi.npz"

if not unreg_phi_path.exists():
    raise FileNotFoundError(f"Missing phi file: {unreg_phi_path}")
if not reg_phi_path.exists():
    raise FileNotFoundError(f"Missing phi file: {reg_phi_path}")

phi_unreg = np.load(unreg_phi_path)["phi"].astype(np.float64)
phi_reg = np.load(reg_phi_path)["phi"].astype(np.float64)

if phi_unreg.shape[0] != locs.shape[0]:
    raise ValueError(
        f"phi_unreg rows ({phi_unreg.shape[0]}) do not match locs rows ({locs.shape[0]})"
    )
if phi_reg.shape[0] != locs.shape[0]:
    raise ValueError(
        f"phi_reg rows ({phi_reg.shape[0]}) do not match locs rows ({locs.shape[0]})"
    )

if phi_unreg.shape[1] < N_BASIS_TO_PLOT or phi_reg.shape[1] < N_BASIS_TO_PLOT:
    raise ValueError(
        f"N_BASIS_TO_PLOT={N_BASIS_TO_PLOT} exceeds available basis columns: "
        f"unreg={phi_unreg.shape[1]}, reg={phi_reg.shape[1]}"
    )

vals = []
for i in range(N_BASIS_TO_PLOT):
    vals.append(phi_unreg[:, i])
    vals.append(phi_reg[:, i])

vals = np.concatenate(vals)
vmax_global = np.max(np.abs(vals))
vmin_global = -vmax_global

for i in range(N_BASIS_TO_PLOT):
    col_unreg = phi_unreg[:, i]
    col_reg = phi_reg[:, i]
    col_diff = col_reg - col_unreg

    plot_basis_map(
        col_unreg,
        locs,
        f"Unreg basis {i+1} (seed {TARGET_SEED})",
        OUT_DIR / f"seed_{TARGET_SEED}_unreg_basis_{i+1}.png",
        vmin=vmin_global,
        vmax=vmax_global,
    )

    plot_basis_map(
        col_reg,
        locs,
        f"Reg basis {i+1} (seed {TARGET_SEED})",
        OUT_DIR / f"seed_{TARGET_SEED}_reg_basis_{i+1}.png",
        vmin=vmin_global,
        vmax=vmax_global,
    )

    vmax_diff = np.max(np.abs(col_diff))
    vmin_diff = -vmax_diff

    plot_basis_map(
        col_diff,
        locs,
        f"Reg - Unreg basis {i+1} (seed {TARGET_SEED})",
        OUT_DIR / f"seed_{TARGET_SEED}_diff_basis_{i+1}.png",
        vmin=vmin_diff,
        vmax=vmax_diff,
    )

print(f"Basis plots saved to: {OUT_DIR}")
print(f"Global basis color scale: [{vmin_global:.6f}, {vmax_global:.6f}]")

np.random.seed(RANDOM_RECON_SEED)
w = np.random.randn(phi_unreg.shape[1])

recon_unreg = phi_unreg @ w
recon_reg = phi_reg @ w
recon_diff = recon_reg - recon_unreg

vmax_recon = max(np.max(np.abs(recon_unreg)), np.max(np.abs(recon_reg)))
vmin_recon = -vmax_recon

plot_basis_map(
    recon_unreg,
    locs,
    f"Unreg reconstructed field (seed {TARGET_SEED})",
    OUT_DIR / f"seed_{TARGET_SEED}_recon_unreg.png",
    vmin=vmin_recon,
    vmax=vmax_recon,
)

plot_basis_map(
    recon_reg,
    locs,
    f"Reg reconstructed field (seed {TARGET_SEED})",
    OUT_DIR / f"seed_{TARGET_SEED}_recon_reg.png",
    vmin=vmin_recon,
    vmax=vmax_recon,
)

vmax_diff = np.max(np.abs(recon_diff))
vmin_diff = -vmax_diff

plot_basis_map(
    recon_diff,
    locs,
    f"Reg - Unreg reconstructed field (seed {TARGET_SEED})",
    OUT_DIR / f"seed_{TARGET_SEED}_recon_diff.png",
    vmin=vmin_diff,
    vmax=vmax_diff,
)

print("Reconstruction plots saved.")
print(f"Reconstruction seed: {RANDOM_RECON_SEED}")